# Semana 8: Evaluación y Validación de Modelos
## Notebook Conceptual (NB1) – Curvas de Aprendizaje y Validación Cruzada

**Propósito:** Consolidar las técnicas de evaluación, diagnóstico de overfitting y selección de hiperparámetros.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Ajustar modelos de distinta complejidad y dibujar curvas de aprendizaje para detectar high bias o high variance.
- Implementar manualmente k-fold cross-validation y comparar con cross_val_score.
- Visualizar el bias-variance tradeoff con modelos polinómicos de diferente grado.
- Usar GridSearchCV y RandomizedSearchCV en Random Forest y comparar resultados.
- Analizar la importancia de la estratificación en clasificación.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.model_selection import learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, make_scorer
from sklearn.datasets import make_classification, make_regression

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Generación de Datos Sintéticos con Ruido

Creamos datos basados en la función seno con ruido para visualizar claramente el efecto de la complejidad del modelo.

In [ ]:
# Función verdadera
def true_function(x):
    return np.sin(2 * np.pi * x)

# Generamos datos
n_samples = 200
X = np.random.uniform(0, 1, n_samples).reshape(-1, 1)
y = true_function(X.ravel()) + np.random.randn(n_samples) * 0.2

# Ordenamos para visualización
X_sorted = np.sort(X, axis=0)
y_true_sorted = true_function(X_sorted.ravel())

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, label='Datos con ruido')
plt.plot(X_sorted, y_true_sorted, 'r-', linewidth=2, label='Función verdadera: sin(2πx)')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos sintéticos para regresión')
plt.legend()
plt.grid(True)
plt.show()

---
## 2. Modelos de Distinta Complejidad

Ajustamos tres modelos de regresión de diferente complejidad:
1. **Lineal** (alto sesgo, baja varianza)
2. **Polinómico grado 5** (complejidad media)
3. **Árbol profundo** (bajo sesgo, alta varianza)

In [ ]:
# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Modelo 1: Regresión Lineal
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr_train = lr.predict(X_train)
y_pred_lr_test = lr.predict(X_test)

# Modelo 2: Polinómico grado 5
poly = PolynomialFeatures(degree=5)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
y_pred_poly_train = lr_poly.predict(X_train_poly)
y_pred_poly_test = lr_poly.predict(X_test_poly)

# Modelo 3: Árbol de decisión profundo
tree = DecisionTreeRegressor(max_depth=15, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree_train = tree.predict(X_train)
y_pred_tree_test = tree.predict(X_test)

# Visualizamos los ajustes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

X_plot = np.linspace(0, 1, 300).reshape(-1, 1)

axes[0].scatter(X_train, y_train, alpha=0.3, label='Train')
axes[0].plot(X_plot, lr.predict(X_plot), 'g-', linewidth=2, label='Lineal')
axes[0].plot(X_sorted, y_true_sorted, 'r--', linewidth=1, label='Verdadera')
axes[0].set_title(f'Lineal\nTrain MSE: {mean_squared_error(y_train, y_pred_lr_train):.4f}\nTest MSE: {mean_squared_error(y_test, y_pred_lr_test):.4f}')
axes[0].legend()

axes[1].scatter(X_train, y_train, alpha=0.3, label='Train')
axes[1].plot(X_plot, lr_poly.predict(poly.transform(X_plot)), 'g-', linewidth=2, label='Polinómico grado 5')
axes[1].plot(X_sorted, y_true_sorted, 'r--', linewidth=1, label='Verdadera')
axes[1].set_title(f'Polinómico grado 5\nTrain MSE: {mean_squared_error(y_train, y_pred_poly_train):.4f}\nTest MSE: {mean_squared_error(y_test, y_pred_poly_test):.4f}')
axes[1].legend()

axes[2].scatter(X_train, y_train, alpha=0.3, label='Train')
axes[2].plot(X_plot, tree.predict(X_plot), 'g-', linewidth=2, label='Árbol profundo')
axes[2].plot(X_sorted, y_true_sorted, 'r--', linewidth=1, label='Verdadera')
axes[2].set_title(f'Árbol profundo\nTrain MSE: {mean_squared_error(y_train, y_pred_tree_train):.4f}\nTest MSE: {mean_squared_error(y_test, y_pred_tree_test):.4f}')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 3. Curvas de Aprendizaje

Las curvas de aprendizaje muestran la evolución del error en entrenamiento y validación al aumentar el tamaño del conjunto de entrenamiento. Son útiles para diagnosticar sesgo y varianza.

In [ ]:
def plot_learning_curve(estimator, X, y, title, ax):
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=5, n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='neg_mean_squared_error'
    )
    
    train_mean = -np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = -np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='red')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Error entrenamiento')
    ax.plot(train_sizes, test_mean, 'o-', color='red', label='Error validación')
    ax.set_xlabel('Tamaño del conjunto de entrenamiento')
    ax.set_ylabel('Error (MSE)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_learning_curve(LinearRegression(), X, y, 'Regresión Lineal (Alto sesgo)', axes[0])

# Para polinómico, necesitamos pipeline
from sklearn.pipeline import Pipeline
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=5)),
    ('linear', LinearRegression())
])
plot_learning_curve(poly_pipeline, X, y, 'Polinómico grado 5 (Complejidad media)', axes[1])

plot_learning_curve(DecisionTreeRegressor(max_depth=15, random_state=42), X, y, 
                    'Árbol profundo (Alta varianza)', axes[2])

plt.tight_layout()
plt.show()

### Interpretación de las curvas de aprendizaje:

- **Regresión Lineal (alto sesgo)**: Ambos errores son altos y convergen rápidamente. Añadir más datos no mejora el modelo.
- **Polinómico grado 5 (complejidad adecuada)**: El error de validación disminuye y se acerca al de entrenamiento. Podría mejorar con más datos.
- **Árbol profundo (alta varianza)**: Gran brecha entre error de entrenamiento (muy bajo) y validación (alto). Añadir más datos podría reducir la brecha.

---
## 4. Implementación Manual de k-Fold Cross-Validation

Implementamos k-fold manualmente y comparamos con `cross_val_score` de sklearn.

In [ ]:
def kfold_manual(X, y, model_class, model_params, k=5):
    """Implementación manual de k-fold CV"""
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]
        
        model = model_class(**model_params)
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        score = mean_squared_error(y_val_fold, y_pred)
        scores.append(score)
    
    return np.array(scores)

# Probamos con regresión lineal
scores_manual = kfold_manual(X, y, LinearRegression, {}, k=5)
print(f"Manual 5-fold MSE: {scores_manual.mean():.4f} (+/- {scores_manual.std()*2:.4f})")

# Con cross_val_score de sklearn
scores_sklearn = cross_val_score(LinearRegression(), X, y, cv=5, scoring='neg_mean_squared_error')
scores_sklearn = -scores_sklearn  # convertir a positivo
print(f"Sklearn 5-fold MSE: {scores_sklearn.mean():.4f} (+/- {scores_sklearn.std()*2:.4f})")

---
## 5. Visualización del Tradeoff Sesgo-Varianza

Usamos modelos polinómicos de diferente grado para observar cómo varían el sesgo y la varianza.

In [ ]:
degrees = [1, 3, 5, 9, 15]
train_errors = []
test_errors = []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(X)
    
    # Validación cruzada para estimar error de prueba
    scores = cross_val_score(LinearRegression(), X_poly, y, cv=10, scoring='neg_mean_squared_error')
    test_errors.append(-scores.mean())
    
    # Error en entrenamiento (con todo el dataset)
    model = LinearRegression()
    model.fit(X_poly, y)
    y_pred = model.predict(X_poly)
    train_errors.append(mean_squared_error(y, y_pred))

plt.figure(figsize=(10, 6))
plt.plot(degrees, train_errors, 'bo-', label='Error entrenamiento')
plt.plot(degrees, test_errors, 'ro-', label='Error validación (CV)')
plt.xlabel('Grado del polinomio')
plt.ylabel('Error (MSE)')
plt.title('Bias-Variance Tradeoff en Modelos Polinómicos')
plt.legend()
plt.grid(True)
plt.show()

---
## 6. GridSearchCV vs RandomizedSearchCV en Random Forest

Comparamos ambos métodos de búsqueda de hiperparámetros en un problema de clasificación.

In [ ]:
# Generamos datos de clasificación
X_clf, y_clf = make_classification(n_samples=500, n_features=10, n_informative=5, 
                                   n_redundant=2, random_state=42)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42
)

In [ ]:
# Definimos el grid de hiperparámetros
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}

# GridSearchCV
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search.fit(X_train_clf, y_train_clf)

print("=== GridSearchCV ===")
print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor accuracy CV: {grid_search.best_score_:.4f}")
print(f"Accuracy en test: {grid_search.score(X_test_clf, y_test_clf):.4f}")

In [ ]:
# RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(
    rf, param_dist, n_iter=30, cv=5, scoring='accuracy', 
    random_state=42, n_jobs=-1, verbose=1
)
random_search.fit(X_train_clf, y_train_clf)

print("\n=== RandomizedSearchCV ===")
print(f"Mejores parámetros: {random_search.best_params_}")
print(f"Mejor accuracy CV: {random_search.best_score_:.4f}")
print(f"Accuracy en test: {random_search.score(X_test_clf, y_test_clf):.4f}")

In [ ]:
# Comparación de tiempos (ya lo veremos en la salida)
# El tiempo de ejecución se puede ver en la salida de cada búsqueda

---
## 7. Importancia de la Estratificación en Clasificación

Demostramos por qué es crucial usar stratified k-fold cuando las clases están desbalanceadas.

In [ ]:
# Generamos datos desbalanceados
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=5, weights=[0.9, 0.1], 
    flip_y=0, random_state=42
)

print(f"Distribución de clases: {np.bincount(y_imb)}")
print(f"Proporción clase 1: {np.mean(y_imb):.2%}")

In [ ]:
# KFold sin estratificar
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kf = []
for train_idx, val_idx in kf.split(X_imb, y_imb):
    prop_val = np.mean(y_imb[val_idx])
    scores_kf.append(prop_val)

# StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skf = []
for train_idx, val_idx in skf.split(X_imb, y_imb):
    prop_val = np.mean(y_imb[val_idx])
    scores_skf.append(prop_val)

print("Proporción de clase 1 en cada pliegue de validación:")
print(f"KFold: {[f'{p:.2%}' for p in scores_kf]}")
print(f"StratifiedKFold: {[f'{p:.2%}' for p in scores_skf]}")
print(f"\nDesviación estándar KFold: {np.std(scores_kf):.4f}")
print(f"Desviación estándar StratifiedKFold: {np.std(scores_skf):.4f}")

---
## 8. Conclusiones

Hemos explorado las técnicas fundamentales de evaluación y validación de modelos:

✔️ **Modelos de distinta complejidad**: Visualizamos underfitting (lineal), ajuste adecuado (polinómico grado 5) y overfitting (árbol profundo).
✔️ **Curvas de aprendizaje**: Diagnosticamos alto sesgo (curvas convergen alto) y alta varianza (gran brecha).
✔️ **k-fold manual vs sklearn**: Verificamos que nuestra implementación coincide con cross_val_score.
✔️ **Bias-variance tradeoff**: Observamos cómo el error de validación primero disminuye y luego aumenta con la complejidad.
✔️ **GridSearch vs RandomSearch**: RandomSearch es más eficiente en espacios grandes.
✔️ **Estratificación**: Es esencial mantener la proporción de clases en cada pliegue cuando hay desbalance.

**Lección clave**: La validación rigurosa es la única manera de estimar correctamente el rendimiento de un modelo y evitar sorpresas en producción.

En el próximo notebook (NB2) aplicaremos estos conceptos a un problema real de predicción de churn.

---
**Fin del Notebook Conceptual - Semana 8**